# Testing Python
* the smallest testable parts of an application, called _units_, are individually and independently scrutinized to ensure they work
* your functions/methods should do ONE thing (and do it well)–testing that thing should be relatively easy to explain
* exercise the __!#@%@!$#__ out of the unit to be sure it works, especially with corner cases, not just the expected cases
* sometimes called "white box testing"
* the overall goal is straightforward:
> write small tests that make behavior clear and help you change code safely


## Key Testing Concepts

* when people say _testing_, they may mean different kinds of tests
* three common categories are:
  * __unit tests__
    * test one small piece of code in isolation
    * usually fast and focused
  * __functional tests__
    * test whether a feature behaves correctly from a user or system point of view
    * often cover more than one function
  * __integration tests__
    * test how multiple parts work together
    * often involve files, databases, APIs, or services
* how to think about the above
  * unit tests ask: _Does this small piece work?_
  * integration tests ask: _Do these parts work together?_
  * functional tests ask: _Does this feature behave correctly?_

In [1]:
def add(a, b):
    return a + b

def apply_discount(price, percent):
    return price * (1 - percent)

print(add(2, 3))
print(apply_discount(100, 0.2))

5
80.0


* in practice, good projects often use a mix of these test types
* different test types give different kinds of confidence

## Test Isolation and Reproducibility

* good tests should be:
  * **isolated**
    * one test should not depend on another test having run first
  * **reproducible**
    * the same test should produce the same result every time when the code has not changed
* if a test depends on shared global state, current time, random values, or external services, it can become fragile

In [2]:
counter = 0

def increment():
    global counter
    
    counter += 1
    return counter

print(increment())
print(increment())

1
2


* code like this can be harder to test because behavior depends on prior state
* tests are usually easier to reason about when each test starts from a known state

## Deterministic vs. Non-Deterministic Behavior
* a **deterministic** function gives the same result for the same input
* a **non-deterministic** function may produce different results even with the same input


In [ ]:
def square(x):
    """Deterministic"""
    return x * x

print(square(5))
print(square(5))

In [1]:
import random

def roll_die():
    """Non-deterministic"""
    return random.randint(1, 6)

print(roll_die())
print(roll_die())

5
6


* testing deterministic code is usually easier
* when code is non-deterministic, common strategies include:
  * controlling randomness with __`random.seed()`__
  * injecting dependencies instead of hard-coding them
  * mocking external systems

In [2]:
import random

random.seed(1) # sequence will now be deterministic
print(roll_die())
print(roll_die())

2
5


## Test-Driven Development (TDD)
![TDD](TDDflowchart.png)
  * **red** = write a failing test
  * **green** = write the simplest code that makes the test pass
  * **refactor** = improve the code while keeping the tests green
* this cycle encourages small steps and clear expectations

## TDD is NOT REALLY ABOUT TESTING!
* traditionally, unit testing is about writing tests to verify the code works…
  * …whereas main focus of TDD is not about testing
  * writing a test before the code is implemented changes the way we think when we implement functionality
  * resulting code is more testable
  * usually simple, elegant design
  * easier to read and maintain
  * why?
  * so really about writing better code, and we get an automated test suite as a nice side effect

### Benefits and Tradeoffs
* benefits
  * clearer requirements
  * smaller design steps
  * better regression protection
  * code that is easier to change
* tradeoffs
  * can feel slower at first
  * not every problem is easiest to discover test-first
  * poorly written tests can become a maintenance burden

## Writing Tests with a Test-First Approach

* a test-first mindset starts by describing expected behavior before implementation
* e.g., suppose we want a function named __`is_even()`__
  * before writing the function, we can write down the expected behavior:
    * __`is_even(2)`__ should be __`True`__
    * __`is_even(3)`__ should be __`False`__
    * __`is_even(0)`__ should be __`True`__
* this pushes us to think clearly about inputs, outputs, and edge cases


In [4]:
# imagine some tests that look like this...

def test_is_even_with_even_number():
    assert is_even(2) is True

def test_is_even_with_odd_number():
    assert is_even(3) is False

def test_is_even_with_zero():
    assert is_even(0) is True

* notice that the tests are:
  * small
  * focused
  * named around behavior
* good pattern to aim for


## Unit Testing with __`pytest`__
* __`pytest`__ is a popular Python testing tool because it's straightforward and expressive
* no boilerplate code needed
* auto-discovers tests which are functions named __`test_*`__
* tests assert the result they expect
   * pytest informs us of failing asserts

### If we name a file __`test_*.py`__, __`pytest`__ will discover it automatically, and run any tests inside which begin with the name __`test_`__

In [12]:
%%writefile test_add.py 

def add(a, b):
    return a + b

def test_add_two_positive_numbers():
    assert add(2, 3) == 5

def test_add_negative_and_positive_number():
    assert add(-1, 4) == 3

Overwriting test_add.py


In [15]:
!pytest

============================= test session starts ==============================
platform darwin -- Python 3.14.0, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/dave-wadestein/Downloads/Python-Intermediate/Intermediate-Python 2026
plugins: anyio-4.11.0, cov-7.1.0
collected 6 items                                                              

test_add.py ..                                                           [ 33%]
test_mean.py ....                                                        [100%]

============================== 6 passed in 1.04s ===============================


In [ ]:
%%writefile test_mean.py
from mean import mean

def test_ints():
    num_list = [1, 2, 3, 4, 5]
    assert mean(num_list) == 3


def test_zero():
    num_list = [0, 2, 4, 6]
    assert mean(num_list) == 3

    
def test_double():
    num_list = [1, 2, 3, 4]
    assert mean(num_list) == 2.5


def test_long():
    big = 100_000_000
    assert mean(range(1, big)) == big / 2.0

In [ ]:
%%writefile mean.py
    
def mean(num_list):
    if len(num_list) == 0:
        raise Exception("Please provide a list of numbers")
    else:
        return sum(num_list) / len(num_list)

* a few important points:
  * no test classes are required
  * plain old __`assert`__ is all we need
  * tests usually read like small examples of expected behavior

### Why plain __`assert`__ is nice
* instead of learning special assertion methods, you can often write exactly what you mean

In [12]:
# what is this actually testing?

def test_string_uppercase():
    assert 'hello'.upper() == 'HELLO' 

## Organizing Tests into Files and Directories
* a common project structure looks like this:

* typical naming conventions:
  * test files often start with __`test_`__
  * test functions often start with __`test_`__
  * keep tests close to the code logically, preferably in a separate __`tests/`__ dir

## Running Tests from the Command Line
* once __`pytest`__ is installed, we saw that we need only invoke it on the command line
  * __`$ pytest`__

* we can also run a single file:
  * __`pytest tests/test_math_utils.py`__

* or even a single test:
  __`pytest tests/test_math_utils.py::test_add_two_positive_numbers`__

## Test Discovery and Organization
* one reason __`pytest`__ feels convenient is that it automatically discovers tests
* by default, it looks for files and functions that follow naming conventions such as:
  * files named __`test_*.py`__
  * functions named __`test_*`__

## Fixtures for Setup and Teardown
* "fixtures" help create reusable setup code for tests
  * instead of repeating the same setup in many tests, you can define a fixture and let tests use it


In [ ]:
import pytest

@pytest.fixture
def sample_list():
    return [1, 2, 3]

def test_list_length(sample_list):
    assert len(sample_list) == 3

def test_list_sum(sample_list):
    assert sum(sample_list) == 6

* often used for
  * sample data
  * temporary files
  * database setup
  * API clients

## Parameterized Tests
* sometimes you want to run the same test logic with many inputs
* parameterized tests make that straightforward

In [8]:
import pytest

@pytest.mark.parametrize(
    'number, expected',
    [
        (2, True),
        (3, False),
        (0, True),
        (-4, True),
    ]
)
def test_is_even(number, expected):
    assert (number % 2 == 0) is expected

* parameterization gives us tests which are
  * shorter
  * easier to extend
  * easier to read as a collection of examples

## Handling Exceptions and Edge Cases
* good tests not only check normal behavior, but also:
  * invalid input
  * boundary cases
  * exceptions
* edge cases often reveal bugs that "happy path" tests miss

#### __`pytest`__ also provides a clean way to test expected exceptions

In [19]:
%%writefile test_div.py

import pytest

def divide(a, b):
    return a / b


def test_divide_by_zero_raises_error():
    with pytest.raises(ZeroDivisionError):
        divide(10, 0)

Overwriting test_div.py


In [20]:
!pytest test_div.py

============================= test session starts ==============================
platform darwin -- Python 3.14.0, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/dave-wadestein/Downloads/Python-Intermediate/Intermediate-Python 2026
plugins: anyio-4.11.0, cov-7.1.0
collected 1 item                                                               

test_div.py F                                                            [100%]

=================================== FAILURES ===================================
_______________________ test_divide_by_zero_raises_error _______________________

    def test_divide_by_zero_raises_error():
        #with pytest.raises(ZeroDivisionError):
>       divide(10, 0)

test_div.py:10: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

a = 10, b = 0

    def divide(a, b):
>       return a / b
               ^^^^^
E       ZeroDivisionError: division by zero

test_div.py:5: ZeroDivisionError
=========================== short test summ

## Code Coverage Analysis
* code coverage measures how much of your code was exercised by tests
* coverage does **not** tell you whether your tests are good, but it can help you identify untested paths
* common goals of coverage analysis:
  * measure test completeness
  * find parts of the code that never ran during testing
  * identify missing branches or edge cases

__`pytest --cov=app`__

__`pytest --cov=app --cov-report=term-missing`__ (to see missed statements/untested paths)

or

__`coverage run -m pytest`__

__`coverage report`__

__`coverage html`__

* the best way to think about coverage:
  * high coverage is useful
  * 100% coverage does not guarantee correctness
  * low coverage often means important behavior is untested

## Example: Untested Code Paths

* consider this function and its tests...

In [56]:
%%writefile test_classify.py
from classify import classify_score

def test_classify_score_a():
    assert classify_score(95) == 'A'


def test_classify_score_b():
    assert classify_score(85) == 'B' 

Overwriting test_classify.py


In [60]:
%%writefile classify.py
    
def classify_score(score):
    if score >= 90:
        return 'A'
    elif score >= 80:
        return 'B'
    elif score >= 70:
        return 'C'
    else:
        return 'F'

Overwriting classify.py


* if your tests only cover __`95`__ and __`85`__, then some paths remain untested
* the Coverage tools help reveal those gaps

In [61]:
# !pip install pytest-cov
!coverage erase; pytest test_classify.py --cov=classify --cov-report=term-missing

============================= test session starts ==============================
platform darwin -- Python 3.14.0, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/dave-wadestein/Downloads/Python-Intermediate/Intermediate-Python 2026
plugins: anyio-4.11.0, cov-7.1.0
collected 2 items                                                              

test_classify.py ..                                                      [100%]

================================ tests coverage ================================
_______________ coverage: platform darwin, python 3.14.0-final-0 _______________

Name          Stmts   Miss  Cover   Missing
-------------------------------------------
classify.py       8      3    62%   7-10
-------------------------------------------
TOTAL             8      3    62%
============================== 2 passed in 0.03s ===============================


In [42]:
!coverage html

Wrote HTML report to ]8;;file:///Users/dave-wadestein/Downloads/Python-Intermediate/Intermediate-Python 2026/htmlcov/index.htmlhtmlcov/index.html]8;;


In [43]:
!open htmlcov/index.html

A stronger test set would also cover:

* a __`C`__ case
* an __`F`__ case
* boundary values such as __`90`__, __`80`__, and __`70`__


## Mini Exercise

* write __`pytest`__ tests for this function 
* include:
  * a normal case
  * an edge case
  * an exception case


In [ ]:
def reciprocal(x):
    if x == 0:
        raise ValueError('x cannot be zero')
    return 1 / x